# HK Free Press News RAG Ingestion with LlamaIndex and ChromaDB

This notebook builds the ingestion side of your news chatbot. It reads text files from `data/hk_free_press_news`, cleans and normalizes each article, splits the articles into retrieval-friendly chunks, embeds those chunks with a local Hugging Face model, and stores them in a persistent ChromaDB collection named `news_chat`.

The flow is:

1. Install and import the required libraries.
2. Point the notebook at your text news folder and ChromaDB folder.
3. Load `.txt` news articles as LlamaIndex documents with useful metadata.
4. Chunk each article into retrieval-friendly nodes.
5. Insert the nodes into ChromaDB.
6. Run a small retrieval test to confirm the vector database is working.

In [ ]:
# Install the RAG ingestion dependencies in the active notebook kernel.
# Run this cell once, then restart the kernel if imports still fail.
%pip install -U llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface chromadb sentence-transformers tqdm psutil arize-phoenix arize-phoenix-otel openinference-instrumentation-llama-index

In [1]:
import re
import sys
import time
import unicodedata
from pathlib import Path

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document, TextNode
from llama_index.vector_stores.chroma import ChromaVectorStore
from tqdm.auto import tqdm

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag.core import create_embedding_model, setup_phoenix_observability
from src.rag.insertion import (
    DEFAULT_INGESTION_OBSERVABILITY_CONFIG,
    ChromaVectorStoreAdmin,
    IngestionPhoenixObserver,
    IngestionVectorIndexer,
    build_ingestion_trace_attributes,
    default_ingestion_run_config,
    upsert_phoenix_project_description,
    list_text_files,
    load_news_documents,
    preview_news_documents,
    build_text_nodes,
    get_chroma_collection
)

## 1. Configure Paths and Embeddings

The news folder is your source corpus. Each `.txt` file is treated as one news article before chunking.

The ChromaDB folder is where the vector database is saved. The collection name now uses a multilingual E5 suffix so bilingual vectors stay separate from the older English-only collection.

This notebook uses `intfloat/multilingual-e5-base` because it supports English and Traditional Chinese with lower memory usage than larger multilingual embedding models.

In [2]:
INGESTION_RUN_CONFIG = default_ingestion_run_config(project_root)
INGESTION_OBSERVABILITY_CONFIG = DEFAULT_INGESTION_OBSERVABILITY_CONFIG

# Insertion-specific defaults live in src/rag/ingestion_config.py.
# The notebook reads them here so this cell stays short and easy to override for experiments.
PROJECT_ROOT = INGESTION_RUN_CONFIG.project_root
paths = INGESTION_RUN_CONFIG.paths
NEWS_DIRS = paths.news_dirs
CHROMA_DIR = paths.chroma_dir
COLLECTION_NAME = INGESTION_RUN_CONFIG.collection_name
CHUNK_SIZE = INGESTION_RUN_CONFIG.chunk_size
CHUNK_OVERLAP = INGESTION_RUN_CONFIG.chunk_overlap
EMBED_MODEL_NAME = INGESTION_RUN_CONFIG.embed_model_name

PHOENIX_ENABLED = INGESTION_OBSERVABILITY_CONFIG.phoenix_enabled
PHOENIX_ENDPOINT = INGESTION_OBSERVABILITY_CONFIG.phoenix_endpoint
PHOENIX_PROJECT_NAME = INGESTION_OBSERVABILITY_CONFIG.phoenix_project_name
PHOENIX_DATASET_NAME = INGESTION_OBSERVABILITY_CONFIG.phoenix_dataset_name
PHOENIX_PROJECT_DESCRIPTION = INGESTION_OBSERVABILITY_CONFIG.phoenix_project_description
PHOENIX_CHUNKING_METHOD = INGESTION_OBSERVABILITY_CONFIG.chunking_method

missing_news_dirs = [news_dir for news_dir in NEWS_DIRS if not news_dir.exists()]
if missing_news_dirs:
    missing_list = ", ".join(str(path) for path in missing_news_dirs)
    raise FileNotFoundError(f"News text folder(s) not found: {missing_list}")

# LlamaIndex uses Settings as global defaults for later index and query operations.
# create_embedding_model centralizes E5 query/passage prefixes so insertion and retrieval match.
Settings.embed_model = create_embedding_model(EMBED_MODEL_NAME)
Settings.llm = None

# Phoenix receives custom ingestion spans plus LlamaIndex spans created later in the notebook.
phoenix_status = setup_phoenix_observability(
    project_name=PHOENIX_PROJECT_NAME,
    endpoint=PHOENIX_ENDPOINT,
    enabled=PHOENIX_ENABLED,
    launch_server=False,
    auto_instrument=False,
    batch=True,
    raise_on_missing=False,
)
ingestion_observer = IngestionPhoenixObserver(
    collection_name=COLLECTION_NAME,
    embedding_model_name=EMBED_MODEL_NAME,
    phoenix_base_url=PHOENIX_ENDPOINT,
    dataset_name=PHOENIX_DATASET_NAME,
    enabled=phoenix_status.enabled,
)
if phoenix_status.enabled:
    upsert_phoenix_project_description(
        base_url=PHOENIX_ENDPOINT,
        project_name=PHOENIX_PROJECT_NAME,
        description=PHOENIX_PROJECT_DESCRIPTION,
    )

# Put ingestion configuration into a parent span so it is easy to read in Phoenix trace attributes.
INGESTION_TRACE_ATTRIBUTES = build_ingestion_trace_attributes(
    config=INGESTION_OBSERVABILITY_CONFIG,
    run_config=INGESTION_RUN_CONFIG,
)

# Daily ChromaDB operations helper: list collections, inspect metadata/samples, and delete collections.
vector_admin = ChromaVectorStoreAdmin(CHROMA_DIR)

print("News source folders:")
for news_dir in NEWS_DIRS:
    print(f"- {news_dir}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"Phoenix status: {phoenix_status.message}")
print(f"Phoenix project: {PHOENIX_PROJECT_NAME}")
print(f"Phoenix dataset name: {PHOENIX_DATASET_NAME}")
print(f"Existing Chroma collections: {vector_admin.list_collection_names()}")

LLM is explicitly disabled. Using MockLLM.


c:\Users\jason\miniconda3\envs\ai\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


OpenTelemetry Tracing Details
|  Phoenix Project: rag-news-ingestion
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix project description updated: rag-news-ingestion
News source folders:
- C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Collection name: run_testing
Embedding model: BAAI/bge-small-en-v1.5
Phoenix status: Phoenix tracing is enabled. Run RAG queries, then open the Phoenix UI to inspect traces.
Phoenix project: rag-news-ingestion
Phoenix dataset name: rag-news-ingestion-chunk-metrics
Existing Chroma collections: ['news

In [3]:
# Daily vector database operations.
# Run this cell whenever you want to inspect ChromaDB before/after ingestion.
print("Collection count:", vector_admin.count_collections())
print("Collection names:", vector_admin.list_collection_names())

if COLLECTION_NAME in vector_admin.list_collection_names():
    summary = vector_admin.describe_collection(COLLECTION_NAME)
    print("Active collection summary:", summary)
    sample = vector_admin.sample_records(COLLECTION_NAME, limit=2)
    print("Sample ids:", sample.get("ids", []))
    print("Sample metadata:", sample.get("metadatas", []))
else:
    print(f"Collection {COLLECTION_NAME!r} does not exist yet. Run the insertion cell to create it.")

# To delete a collection intentionally, uncomment the next line.
# vector_admin.delete_collection("collection_name_to_delete", missing_ok=True)

Collection count: 1
Collection names: ['news_chat_multilingual_e5_base']
Collection 'run_testing' does not exist yet. Run the insertion cell to create it.


## 2. Load Text News Articles

Your new dataset is already stored as text files, so we do not need PDF extraction anymore. That is a good improvement for RAG: plain text is usually cleaner, faster to load, and less likely to produce unreadable characters than PDF parsing.

In this section, each `.txt` file becomes one LlamaIndex `Document`. The loader also extracts useful metadata from the file name:

- `article_date`: the date prefix at the start of the file name, such as `01-01-2026`
- `article_title`: the article title after the underscore
- `file_name` and `file_path`: the original source file information

This metadata is stored alongside each chunk in ChromaDB, so later retrieval can show where an answer came from.

In [4]:
text_files = list_text_files(NEWS_DIRS)
print(f"Found {len(text_files)} text news file(s) across {len(NEWS_DIRS)} source folder(s).")
for text_file in text_files[:10]:
    print(f"- {text_file.name}")

if len(text_files) > 10:
    print(f"... and {len(text_files) - 10} more")


# Trace loading latency because it is runtime behavior; store per-document quality metrics as dataset records.
documents, document_load_latency = ingestion_observer.measure(
    "ingestion.load_documents",
    lambda: load_news_documents(NEWS_DIRS, show_progress=True),
    {"files.discovered": len(text_files)},
)
document_metrics = ingestion_observer.build_document_metrics(documents)
print(f"Loaded {len(documents)} news article document(s) in {document_load_latency:.2f}s.")
print(f"Prepared {len(document_metrics)} document metric record(s) for Phoenix dataset upload.")
preview_news_documents(documents)

Found 1076 text news file(s) across 1 source folder(s).
- 01-01-2026_Chinese homeschooled students embrace freer youth in cutthroat market.txt
- 01-01-2026_Explainer How deadly Tai Po fire brings to light bid-rigging epidemic in Hong Kong renovation industry.txt
- 01-01-2026_Listed companies face a new year’s carbon resolution – let’s join them.txt
- 01-01-2026_Taiwan’s president vows to defend sovereignty after China drills.txt
- 01-02-2026_Chinese cash in jewellery at automated gold recyclers as prices soar.txt
- 01-02-2026_UK Prime Minister Keir Starmer ends China trip aimed at reset despite Trump warning.txt
- 01-02-2026_‘Is there a choice or no choice’ Gov’t handling of long-term housing frustrates Tai Po fire survivors.txt
- 01-03-2026_Iran’s Ayatollah Ali Khamenei killed in massive US and Israeli attack.txt
- 01-03-2026_Tai Po fire Some survivors wary over resettlement plan after ‘ignored’ calls to rebuild on site.txt
- 01-03-2026_‘Unofficial’ talks on plastic pollution treaty t

Loading and cleaning articles:   0%|          | 0/1076 [00:00<?, ?file/s]

- 17-01-2026_HKFP Monitor Jan 17, 2026 Jimmy Lai trial ignites courtroom queue rivalry.txt
- 31-01-2026_HKFP Monitor Jan 31, 2026 Democrats outside bars; lawmaker in hot water.txt
Loaded 1074 news article document(s) in 11.71s.
Prepared 1074 document metric record(s) for Phoenix dataset upload.

--- Preview 1 ---
Source folder: hk_free_press_news
Date: 01-01-2026
Title: Chinese homeschooled students embrace freer youth in cutthroat market
By Mary Yang

Fourteen-year-old Estella spends her weekdays studying Spanish, rock climbing or learning acupuncture in her living room as part of her homeschooling since she left China’s gruelling public school system.

Her parents withdrew her from her Shanghai school three years ago, worried she was struggling to keep up with a demanding curriculum they believe will soon be outdated in the era of artificial intelligence (AI).

They are among a small number of parents in China who are rethinking the country’s rigorous education system, in which schoo

## 3. Chunk the News Text

RAG works better when articles are split into chunks that are large enough to contain meaning but small enough to retrieve precisely.

Here we use sentence-aware chunking with overlap. The overlap is important for news articles because names, places, and quoted context may span chunk boundaries. Each chunk keeps the article metadata from the original text file, so retrieved chunks can still show the article date, title, and file name.

In [5]:
# Chunking latency is a span metric; chunk counts per document are also written into dataset records.
nodes, chunking_latency = ingestion_observer.measure(
    "ingestion.chunk_documents",
    lambda: build_text_nodes(
        documents,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        show_progress=True,
    ),
    {
        "documents.count": len(documents),
        "chunk.size": CHUNK_SIZE,
        "chunk.overlap": CHUNK_OVERLAP,
    },
)
ingestion_observer.record_chunking(documents, nodes, chunking_latency)

print(f"Loaded {len(documents)} news article document(s).")
print(f"Created {len(nodes)} text chunk(s) in {chunking_latency:.2f}s.")
print("\nPreview of first chunk:")
print(nodes[0].get_content()[:800] if nodes else "No chunks created.")

Annotating chunk metadata:   0%|          | 0/1591 [00:00<?, ?chunk/s]

Loaded 1074 news article document(s).
Created 1591 text chunk(s) in 3.26s.

Preview of first chunk:
By Mary Yang

Fourteen-year-old Estella spends her weekdays studying Spanish, rock climbing or learning acupuncture in her living room as part of her homeschooling since she left China’s gruelling public school system.

Her parents withdrew her from her Shanghai school three years ago, worried she was struggling to keep up with a demanding curriculum they believe will soon be outdated in the era of artificial intelligence (AI).

They are among a small number of parents in China who are rethinking the country’s rigorous education system, in which school days can last 10 hours, with students often working late into the evening on extra tutoring and homework.

“In the future, education models and jobs will face huge changes due to AI,” Estella’s mother Xu Zoe told AFP, using a pseudonym.

“We


In [6]:
RESET_COLLECTION = True

index_builder = IngestionVectorIndexer(
    observer=ingestion_observer,
    news_dirs=NEWS_DIRS,
    embed_model_name=EMBED_MODEL_NAME,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Group all ingestion spans under one parent span so Phoenix shows one run trace with child spans.
with ingestion_observer.span("ingestion.run", INGESTION_TRACE_ATTRIBUTES):
    index = index_builder.build_or_update_vector_index(
        nodes,
        chroma_dir=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        reset_collection=RESET_COLLECTION,
        show_progress=True,
    )
    collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
    dataset_upload_result = ingestion_observer.export_document_metrics_dataset()
    ingestion_summary = ingestion_observer.summary()

print(f"Collection name: {COLLECTION_NAME}")
print(f"Stored vector count: {collection.count()}")
print(f"Collection metadata: {vector_admin.get_collection_metadata(COLLECTION_NAME)}")
print(f"Ingestion summary: {ingestion_summary}")
print(f"Phoenix chunk metrics dataset action: {ingestion_observer.last_dataset_export_action}")
if ingestion_observer.last_dataset_export_error:
    print(f"Phoenix dataset export error: {ingestion_observer.last_dataset_export_error}")
print("Phoenix chunk metrics dataset upload:", "ok" if dataset_upload_result else "not available")

Collection did not exist yet: run_testing


Embedding and writing Chroma batches:   0%|          | 0/7 [00:00<?, ?batch/s]

Collection name: run_testing
Stored vector count: 1591
Collection metadata: {'embedding_model': 'BAAI/bge-small-en-v1.5', 'chunk_size': 800, 'source_folders': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk_free_press_news', 'chunk_overlap': 120, 'source_folder_count': 1}
Ingestion summary: {'documents': 1074, 'chunks': 1591, 'invalid_metadata_documents': 0, 'total_tokens_ingested_estimate': 681245, 'vector_batches': 7, 'vectors_written': 1591, 'average_vectors_per_second': 6.357034068876607, 'ram_mb_current': 1076.35546875}
Phoenix chunk metrics dataset action: created
Phoenix chunk metrics dataset upload: ok


## 4. Insert Chunks into ChromaDB

This cell creates a persistent ChromaDB collection and inserts the chunks through LlamaIndex. `RESET_COLLECTION = True` rebuilds this notebook's collection cleanly, which prevents duplicate chunks when you rerun the notebook. Set it to `False` only when you intentionally want to append new chunks.

In [ ]:
RESET_COLLECTION = True

index_builder = IngestionVectorIndexer(
    observer=ingestion_observer,
    news_dirs=NEWS_DIRS,
    embed_model_name=EMBED_MODEL_NAME,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Group all ingestion spans under one parent span so Phoenix shows one run trace with child spans.
with ingestion_observer.span("ingestion.run", INGESTION_TRACE_ATTRIBUTES):
    index = index_builder.build_or_update_vector_index(
        nodes,
        chroma_dir=CHROMA_DIR,
        collection_name=COLLECTION_NAME,
        reset_collection=RESET_COLLECTION,
        show_progress=True,
    )
    collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
    dataset_upload_result = ingestion_observer.export_document_metrics_dataset()
    ingestion_summary = ingestion_observer.summary()

print(f"Collection name: {COLLECTION_NAME}")
print(f"Stored vector count: {collection.count()}")
print(f"Collection metadata: {vector_admin.get_collection_metadata(COLLECTION_NAME)}")
print(f"Ingestion summary: {ingestion_summary}")
print(f"Phoenix chunk metrics dataset action: {ingestion_observer.last_dataset_export_action}")
if ingestion_observer.last_dataset_export_error:
    print(f"Phoenix dataset export error: {ingestion_observer.last_dataset_export_error}")
print("Phoenix chunk metrics dataset upload:", "ok" if dataset_upload_result else "not available")

Deleted existing collection: run_testing


Embedding and writing Chroma batches:   0%|          | 0/7 [00:00<?, ?batch/s]

Collection name: run_testing
Stored vector count: 1591
Collection metadata: {'chunk_size': 800, 'source_folder_count': 1, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'source_folders': 'C:\\Program Files\\Studying\\coding\\RAG_project\\data\\hk_free_press_news', 'chunk_overlap': 120}
Ingestion summary: {'documents': 1074, 'chunks': 1591, 'invalid_metadata_documents': 0, 'total_tokens_ingested_estimate': 681245, 'vector_batches': 14, 'vectors_written': 3182, 'average_vectors_per_second': 6.704607008075238, 'ram_mb_current': 1108.15625}
Phoenix chunk metrics dataset action: created
Phoenix chunk metrics dataset upload: ok
